# TRINITY quickstart

This notebook reads a TRINITY run and plots it. A finished run ships with the
repository under `examples/lifecycle_run/`, so the notebook works immediately
after cloning — you do not need to run a simulation first.

It is a tracked example config:

```bash
python run.py param/cloud_example_homogeneous.param
```

a $10^6\,M_\odot$ uniform cloud at 1% star-formation efficiency — chosen because it
runs the full lifecycle, from the energy-driven bubble through the transition to a
momentum-driven shell and a stopping fate. Run your own and point `RUN` at it to see
your results instead.

The shipped copy keeps every 4th snapshot of the original run, to stay small enough
to live in a git repository. Nothing inside a snapshot was altered.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from trinity._output.trinity_reader import TrinityOutput

# The reader wants the .jsonl file; it picks up metadata.json from the same
# folder automatically, which is why a run directory travels as a whole.
RUN = Path('lifecycle_run')

output = TrinityOutput.open(RUN / 'dictionary.jsonl')
output

## What is in the file

`info()` prints the run's context. Pass `verbose=True` to list every key with its
description and original units — useful when you are looking for a quantity and
do not know its name.

In [ ]:
output.info()

## Shell radius and velocity

`get()` pulls any key across every snapshot as a numpy array. Time is in Myr,
radius in pc, velocity in pc/Myr (1 pc/Myr ≈ 0.98 km/s).

The shaded bands mark the evolutionary phases. Note the wrinkle: the
energy-driven regime covers **two** phase labels, `energy` and `implicit`, so
`filter(phase='energy')` alone would give you only its first half.

In [ ]:
t  = output.get('t_now')
R2 = output.get('R2')
v2 = output.get('v2')
phase = np.array(output.get('current_phase', as_array=False))

PHASE_BANDS = {
    'energy-driven': (('energy', 'implicit'), '#DCEAF5'),
    'transition':    (('transition',),        '#F5E9DC'),
    'momentum':      (('momentum',),          '#E6F0E4'),
}

def shade(ax):
    """Shade the time axis by evolutionary phase."""
    for label, (names, colour) in PHASE_BANDS.items():
        mask = np.isin(phase, names)
        if not mask.any():
            continue
        ax.axvspan(t[mask].min(), t[mask].max(), color=colour, zorder=0, label=label)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 6), sharex=True)

shade(ax1)
ax1.plot(t, R2, color='#1E2430', lw=1.6)
ax1.set_ylabel('shell radius $R_2$  [pc]')
ax1.set_yscale('log')
ax1.legend(loc='lower right', fontsize=9, frameon=False)

shade(ax2)
ax2.plot(t, v2, color='#0EA5C8', lw=1.6)
ax2.set_ylabel('velocity $v_2$  [pc/Myr]')
ax2.set_xlabel('time  [Myr]')
ax2.set_yscale('log')

for ax in (ax1, ax2):
    ax.set_xscale('log')

fig.suptitle(f'{output.model_name}: shell expansion', y=0.94)
fig.tight_layout()

## The force budget

Every snapshot carries the forces acting on the shell, so you can see which
feedback channel is doing the work at any moment. `F_grav` pulls inward; the
rest push out.

In [ ]:
FORCES = {
    'F_ram':  'wind ram pressure',
    'F_HII':  'photoionised gas',
    'F_rad':  'radiation pressure',
    'F_grav': 'gravity (inward)',
}

fig, ax = plt.subplots(figsize=(7, 4))
shade(ax)

for key, label in FORCES.items():
    f = np.abs(output.get(key))
    style = '--' if key == 'F_grav' else '-'
    ax.plot(t, f, style, lw=1.5, label=label)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('time  [Myr]')
ax.set_ylabel(r'|force|  [$M_\odot$ pc Myr$^{-2}$]')
ax.legend(fontsize=9, frameon=False, ncol=2)
ax.set_title('What drives the shell')
fig.tight_layout()

## A radial profile

Most keys are single numbers per snapshot, but a few are 1-D profiles. Each one
carries its own radius axis (`*_r_arr`), and those spanning many decades are
stored as $\log_{10}$ (the `log_` prefix). Here is the shell density at a few
times.

In [ ]:
# Early snapshots predate a resolved shell, so their profile arrays are nearly
# empty. Pick from the snapshots that actually carry one.
usable = [s for s in output if len(s.get('shell_r_arr', []) or []) >= 10]
snapshots = [usable[i] for i in np.linspace(0, len(usable) - 1, 4, dtype=int)]

fig, ax = plt.subplots(figsize=(7, 4))

for snap in snapshots:
    r = np.asarray(snap['shell_r_arr'])
    n = 10 ** np.asarray(snap['log_shell_n_arr'])  # stored in log10
    if r.size:
        ax.plot(r, n, lw=1.4, label=f"t = {snap.t_now:.3g} Myr")

ax.set_xlabel('radius  [pc]')
ax.set_ylabel(r'shell number density  [cm$^{-3}$]')
ax.set_yscale('log')
ax.legend(fontsize=9, frameon=False)
ax.set_title('Shell density profile')
fig.tight_layout()

## How the run ended

In [ ]:
print(output.termination)
print()
print({k: output.final_state[k] for k in ('t_now', 'R2', 'v2') if k in output.final_state})

## Where to go next

- `output.get_at_time(t)` — the state at any time, interpolated.
- `output.filter(phase=..., t_min=..., t_max=...)` — a sub-range, itself a
  `TrinityOutput`.
- `output.to_dataframe()` — the whole run as a pandas `DataFrame`.
- `output.info(verbose=True)` — every available key, described.

Parameter names and units are listed in the
[parameter reference](https://jiaweiteh.github.io/trinity-web/?view=docs&page=parameters);
the run and sweep workflow is on the
[running page](https://jiaweiteh.github.io/trinity-web/?view=docs&page=running).